# Домашнє завдання: Прогнозування орендної плати за житло

## Мета завдання
Застосувати знання з лекції для побудови моделі лінійної регресії, що прогнозує орендну плату за житло в Індії. Ви пройдете весь цикл вирішення задачі машинного навчання: від дослідницького аналізу до оцінки якості моделі.

## Опис датасету
**House Rent Prediction Dataset** містить інформацію про 4700+ оголошень про оренду житла в Індії з такими параметрами:
- **BHK**: Кількість спалень, залів, кухонь
- **Rent**: Орендна плата (цільова змінна)
- **Size**: Площа в квадратних футах
- **Floor**: Поверх та загальна кількість поверхів
- **Area Type**: Тип розрахунку площі
- **Area Locality**: Район
- **City**: Місто
- **Furnishing Status**: Стан меблювання
- **Tenant Preferred**: Тип орендаря
- **Bathroom**: Кількість ванних кімнат
- **Point of Contact**: Контактна особа

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `House_Rent_Dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (включно з типами даних та кількістю значень)


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
df = pd.read_csv('House_Rent_Dataset.csv', sep=None, engine='python')

In [4]:
df.shape

(4746, 12)

In [5]:
df.head(5)

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


## Завдання 2: Дослідницький аналіз даних (EDA) (5 балів)

**Що потрібно зробити:**
1. **Аналіз пропущених значень.** Перевірте наявність і відсоток пропущених значень у кожній колонці
2. **Базова статистика.** Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.
3. **Аналіз цільової змінної.** Побудуйте гістограму розподілу цільової змінної (Rent)
4. **Робота з викидами.** Знайдіть та видаліть викиди в цільовій змінній (якщо є). Визначити викиди можна будь-яким зрозумілим для вас способом, як варіант - таким, що використовується в побудові box-plot (https://en.wikipedia.org/wiki/Box_plot#Example_with_outliers).
5. **Аналіз категоріальних змінних.** Виведіть кількість унікальних значень для кожної з категоріальних колонок.


In [7]:
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_percent

,0
Posted On,0.0
BHK,0.0
Rent,0.0
Size,0.0
Floor,0.0
Area Type,0.0
Area Locality,0.0
City,0.0
Furnishing Status,0.0
Tenant Preferred,0.0


In [8]:
df.describe().round(2)

,BHK,Rent,Size,Bathroom
count,4746.00,4746.00,4746.00,4746.00
mean,2.08,34993.45,967.49,1.97
std,0.83,78106.41,634.20,0.88
min,1.00,1200.00,10.00,1.00
25%,2.00,10000.00,550.00,1.00
50%,2.00,16000.00,850.00,2.00
75%,3.00,33000.00,1200.00,2.00
max,6.00,3500000.00,8000.00,10.00


In [9]:
# Розподіл
fig = px.histogram(
    df,
    x='Rent',
    nbins=100,
    title='Розподіл цільової змінної (Оренда)',
    labels={'Rent': 'Орендна плата'}
)
fig.update_layout(
    showlegend=False,
    height=400
    )
fig.update_layout(
    yaxis_title='Кількість оголошень')
fig.show()

In [10]:
Q1 = df['Rent'].quantile(0.25)
Q3 = df['Rent'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['Rent'] >= lower_bound) & (df['Rent'] <= upper_bound)]

print("Cleaned DataFrame:")
print(df_clean)

Cleaned DataFrame:
       Posted On  BHK   Rent  Size            Floor    Area Type  \
0     2022-05-18    2  10000  1100  Ground out of 2   Super Area   
1     2022-05-13    2  20000   800       1 out of 3   Super Area   
2     2022-05-16    2  17000  1000       1 out of 3   Super Area   
3     2022-07-04    2  10000   800       1 out of 2   Super Area   
4     2022-05-09    2   7500   850       1 out of 2  Carpet Area   
...          ...  ...    ...   ...              ...          ...   
4741  2022-05-18    2  15000  1000       3 out of 5  Carpet Area   
4742  2022-05-15    3  29000  2000       1 out of 4   Super Area   
4743  2022-07-10    3  35000  1750       3 out of 5  Carpet Area   
4744  2022-07-06    3  45000  1500     23 out of 34  Carpet Area   
4745  2022-05-04    2  15000  1000       4 out of 5  Carpet Area   

                 Area Locality       City Furnishing Status  Tenant Preferred  \
0                       Bandel    Kolkata       Unfurnished  Bachelors/Family   
1 

In [11]:
df_clean.Rent.describe().round(2)

,Rent
count,4226.00
mean,19286.16
std,13825.40
min,1200.00
25%,9500.00
50%,15000.00
75%,25000.00
max,67000.00


In [12]:
fig = px.histogram(
    df_clean,
    x='Rent',
    nbins=100,
    title='Розподіл цільової змінної (Оренда)',
    labels={'Rent': 'Орендна плата'}
)
fig.update_layout(
    showlegend=False,
    height=400
    )
fig.update_layout(
    yaxis_title='Кількість оголошень')
fig.show()

In [13]:
df_clean['Rent_log'] = np.log(df['Rent'])  #log because no 0 values, not log1p

# Plot the transformed histogram
fig = px.histogram(df_clean, x='Rent_log', nbins=100, title='Log-transformed Distribution of Rent')
fig.show()

/tmp/ipykernel_1878/1934792015.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
# for categorical values
for column in ['Floor', 'Area Type', 'Area Locality', 'City', 'Furnishing Status', 'Tenant Preferred', 'Point of Contact']:
    print(f"Column: {column}")
    print("Unique values:", df_clean[column].unique())
    print("N of unique values:", df_clean[column].nunique())
    print("Mode:", df_clean[column].mode().iloc[0])
    print("-" * 30)

# in cleaned data the most frequent city changed

Column: Floor
Unique values: ['Ground out of 2' '1 out of 3' '1 out of 2' 'Ground out of 1'
 'Ground out of 4' '1 out of 4' '1 out of 1' 'Ground out of 3'
 '2 out of 3' '4 out of 5' '2 out of 2' '2 out of 5' '4 out of 14'
 '3 out of 3' '5 out of 5' '4 out of 4' '7 out of 8' '2 out of 4'
 '3 out of 4' '1 out of 5' '8 out of 5' 'Ground out of 6' '2 out of 1'
 'Upper Basement out of 4' 'Ground out of 5' '3 out of 5' '11 out of 19'
 '5 out of 10' '11 out of 14' 'Lower Basement out of 2' '2 out of 7'
 '7 out of 10' '6 out of 7' '4 out of 7' '2 out of 8' '5 out of 12'
 '11 out of 21' '14 out of 23' '9 out of 20' 'Upper Basement out of 9'
 '19 out of 24' '3 out of 21' '1 out of 22' '3 out of 7' '8 out of 8'
 '6 out of 12' 'Upper Basement out of 16' '34 out of 48' '5 out of 8'
 '5 out of 14' '14 out of 40' '9 out of 22' '12 out of 18' '26 out of 44'
 '1 out of 8' 'Ground out of 7' '13 out of 20' '16 out of 23'
 '10 out of 18' '10 out of 32' '12 out of 24' '3 out of 30' '13 out of 21'
 '13 out 


## Завдання 3: Аналіз кореляцій та взаємозв'язків (3 бали)

**Що потрібно зробити:**
1. Обчисліть матрицю кореляцій для числових змінних
2. Візуалізуйте кореляційну матрицю за допомогою heatmap
3. Побудуйте scatter plot між Size та Rent
4. Проаналізуйте взаємозв'язок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)


In [15]:
metrics_df = df_clean[['BHK', 'Rent',	'Size',	'Bathroom', 'Rent_log']].dropna()

# Матриця кореляцій
correlation_matrix = metrics_df.corr()

# Візуалізація кореляцій
fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Кореляція між метриками',
    labels=dict(color="Кореляція")
)
fig.update_layout(height=500)
fig.show()

In [20]:
fig = px.scatter(df_clean,
                 x='Size', y='Rent_log',
                 title="Зв'язок між площею та log-орендною платою",
                 labels={'Size': 'Площа', 'Rent_log': 'log-Орендна плата'},
                 trendline="ols")
fig.update_traces(line=dict(color='red'), selector=dict(mode='lines'))
fig.show()

In [21]:
fig = px.scatter(df_clean,
                 x='Size', y='Rent',
                 title="Зв'язок між площею та орендною платою",
                 labels={'Size': 'Площа', 'Rent': 'Орендна плата'},
                 trendline="ols")
fig.update_traces(line=dict(color='red'), selector=dict(mode='lines'))
fig.show()

## Завдання 4: Feature Engineering та підготовка даних (4 бали)

**Що потрібно зробити:**
1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding. Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками. Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).
2. **Опціонально (по 0.5 бала за кожну доцільну ознаку):** Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі
3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.
4. Розділіть дані на ознаки (X) та цільову змінну (y)
5. Застосуйте стандартизацію до числових ознак


In [23]:
df_clean['Posted On'] = pd.to_datetime(df_clean['Posted On'], format='%Y-%m-%d')

# Create the 'Season' variable based on the month from 'Posted On'
def get_season(month):
    if month in [12, 1, 2]:  # Winter
        return 'Winter'
    elif month in [3, 4, 5]:  # Summer
        return 'Summer'
    elif month in [6, 7, 8, 9]:  # Monsoon
        return 'Monsoon'
    else:  # Post-Monsoon (Oct-Nov)
        return 'Post-Monsoon'

# Apply function to create the 'Season' column
df_clean['Season'] = df_clean['Posted On'].dt.month.apply(get_season)

df_clean.head()

/tmp/ipykernel_1878/89161925.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_1878/89161925.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,Rent_log,Season
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner,9.210340,Summer
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,9.903488,Summer
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,9.740969,Summer
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner,9.210340,Monsoon
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner,8.922658,Summer


In [33]:
# Add categories for floor types based on the floor ratio
def categorize_floor(floor_info):
    # Extract floor and total floors from the 'Floor' column
    if 'out of' in floor_info:
        floor, total_floors = floor_info.split(' out of ')

        # Treat 'Ground', 'Upper Basement', and 'Lower Basement' as 0
        if 'Ground' in floor_info or 'Upper Basement' in floor_info or 'Lower Basement' in floor_info:
            floor = '0'  # Treat Ground, Upper Basement, and Lower Basement as floor 0

        floor = int(floor)  # Convert floor to integer
        total_floors = int(total_floors)

        # Calculate the ratio of the current floor to the total floors
        ratio = floor / total_floors

        # Categorize based on ratio
        if 'Ground' in floor_info:
            return 'Ground'
        elif 'Upper Basement' in floor_info or 'Lower Basement' in floor_info:
            return 'Basement'
        elif ratio < 0.25:  # Low floor
            return 'Low'
        elif ratio < 0.75:  # Medium floor
            return 'Medium'
        else:  # High floor
            return 'High'
    else:
        return 'Unknown'

# Apply the categorization function
df_clean['Floor Category'] = df_clean['Floor'].apply(categorize_floor)


/tmp/ipykernel_1878/3861080909.py:32: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [34]:
df_clean['Floor Category'].unique()

array(['Ground', 'Medium', 'High', 'Low', 'Basement', 'Unknown'],
      dtype=object)

In [35]:
df_clean[df_clean['Floor Category'] == 'Unknown']


,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,Rent_log,Season,Floor Category
2553,2022-06-18,2,20000,400,3,Super Area,"Kasturba Niketan, Lajpat Nagar 2",Delhi,Unfurnished,Bachelors/Family,1,Contact Owner,9.903488,Monsoon,Unknown
2883,2022-05-23,1,18000,450,Ground,Carpet Area,"DDA Flat AD Block, Shalimar Bagh AD Block",Delhi,Furnished,Bachelors/Family,1,Contact Owner,9.798127,Summer,Unknown
4490,2022-06-12,3,15000,900,1,Super Area,"Malakpet, NH 9",Hyderabad,Semi-Furnished,Bachelors/Family,3,Contact Owner,9.615805,Monsoon,Unknown
4560,2022-05-31,3,15000,1270,1,Carpet Area,Tarnaka,Hyderabad,Furnished,Family,2,Contact Owner,9.615805,Summer,Unknown


In [36]:
# Manually change the category
df_clean.loc[2883, 'Floor Category'] = 'Ground'

# Verify the update
print(df_clean.loc[2883])

Posted On                                  2022-05-23 00:00:00
BHK                                                          1
Rent                                                     18000
Size                                                       450
Floor                                                   Ground
Area Type                                          Carpet Area
Area Locality        DDA Flat AD Block, Shalimar Bagh AD Block
City                                                     Delhi
Furnishing Status                                    Furnished
Tenant Preferred                              Bachelors/Family
Bathroom                                                     1
Point of Contact                                 Contact Owner
Rent_log                                              9.798127
Season                                                  Summer
Floor Category                                          Ground
Name: 2883, dtype: object


In [37]:
df_clean['Floor Category'].unique()

array(['Ground', 'Medium', 'High', 'Low', 'Basement', 'Unknown'],
      dtype=object)

In [53]:
# One-Hot encode the 'Floor Category' variable
df_encoded = pd.get_dummies(df_clean, columns=['Floor Category', 'Area Type', 'City', 'Furnishing Status', 'Point of Contact', 'Season'], drop_first=False)

df_encoded_clean = df_encoded.loc[:, ~df_encoded.columns.isin(df_clean.columns)]

# Final cleaned DataFrame with all features
df_final = pd.concat([df_clean, df_encoded_clean], axis=1)

df_final.columns

Index(['Posted On', 'BHK', 'Rent', 'Size', 'Floor', 'Area Type',
       'Area Locality', 'City', 'Furnishing Status', 'Tenant Preferred',
       'Bathroom', 'Point of Contact', 'Rent_log', 'Season', 'Floor Category',
       'Floor Category_Basement', 'Floor Category_Ground',
       'Floor Category_High', 'Floor Category_Low', 'Floor Category_Medium',
       'Floor Category_Unknown', 'Area Type_Built Area',
       'Area Type_Carpet Area', 'Area Type_Super Area', 'City_Bangalore',
       'City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Kolkata',
       'City_Mumbai', 'Furnishing Status_Furnished',
       'Furnishing Status_Semi-Furnished', 'Furnishing Status_Unfurnished',
       'Point of Contact_Contact Agent', 'Point of Contact_Contact Builder',
       'Point of Contact_Contact Owner', 'Season_Monsoon', 'Season_Summer'],
      dtype='object')

In [55]:
columns_to_select = [
    'Rent_log', 'BHK', 'Size', 'Floor Category_Basement', 'Floor Category_Ground',
       'Floor Category_High', 'Floor Category_Low', 'Floor Category_Medium',
       'Floor Category_Unknown', 'Area Type_Built Area',
       'Area Type_Carpet Area', 'Area Type_Super Area', 'City_Bangalore',
       'City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Kolkata',
       'City_Mumbai', 'Furnishing Status_Furnished',
       'Furnishing Status_Semi-Furnished', 'Furnishing Status_Unfurnished',
       'Point of Contact_Contact Agent', 'Point of Contact_Contact Builder',
       'Point of Contact_Contact Owner', 'Season_Monsoon', 'Season_Summer'
]

# Calculate the correlation matrix for the selected columns
correlation_matrix = df_final[columns_to_select].corr()

# Візуалізація кореляцій
fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Кореляційна матриця всіх числових ознак',
    labels=dict(color="Кореляція")
)
fig.update_layout(height=500)
fig.show()

In [56]:
df= df_final[columns_to_select]
df

,Rent_log,BHK,Size,Floor Category_Basement,Floor Category_Ground,Floor Category_High,Floor Category_Low,Floor Category_Medium,Floor Category_Unknown,Area Type_Built Area,...,City_Kolkata,City_Mumbai,Furnishing Status_Furnished,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Point of Contact_Contact Agent,Point of Contact_Contact Builder,Point of Contact_Contact Owner,Season_Monsoon,Season_Summer
0,9.210340,2,1100,False,True,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,True
1,9.903488,2,800,False,False,False,False,True,False,False,...,True,False,False,True,False,False,False,True,False,True
2,9.740969,2,1000,False,False,False,False,True,False,False,...,True,False,False,True,False,False,False,True,False,True
3,9.210340,2,800,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,True,False
4,8.922658,2,850,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4741,9.615805,2,1000,False,False,False,False,True,False,False,...,False,False,False,True,False,False,False,True,False,True
4742,10.275051,3,2000,False,False,False,False,True,False,False,...,False,False,False,True,False,False,False,True,False,True
4743,10.463103,3,1750,False,False,False,False,True,False,False,...,False,False,False,True,False,True,False,False,True,False
4744,10.714418,3,1500,False,False,False,False,True,False,False,...,False,False,False,True,False,True,False,False,True,False


In [57]:
correlation_matrix['Rent_log'].sort_values()

,Rent_log
Point of Contact_Contact Owner,-0.511422
Area Type_Super Area,-0.291464
City_Kolkata,-0.281586
Furnishing Status_Unfurnished,-0.226768
Floor Category_Ground,-0.226619
Season_Summer,-0.117033
City_Bangalore,-0.072471
City_Chennai,-0.070308
City_Hyderabad,-0.049293
Point of Contact_Contact Builder,-0.023864


In [59]:
top_corr = correlation_matrix['Rent_log'].abs().sort_values(ascending=False)
top_corr_features = top_corr[top_corr > 0.01].index
top_corr_features

Index(['Rent_log', 'Point of Contact_Contact Agent',
       'Point of Contact_Contact Owner', 'BHK', 'City_Mumbai', 'Size',
       'Area Type_Carpet Area', 'Area Type_Super Area', 'City_Kolkata',
       'Furnishing Status_Unfurnished', 'Floor Category_Ground',
       'Floor Category_Low', 'Furnishing Status_Furnished',
       'Furnishing Status_Semi-Furnished', 'Season_Summer', 'Season_Monsoon',
       'Floor Category_Medium', 'City_Bangalore', 'City_Chennai',
       'City_Hyderabad', 'Point of Contact_Contact Builder',
       'Floor Category_Basement', 'City_Delhi', 'Floor Category_High',
       'Area Type_Built Area'],
      dtype='object')

In [62]:
features = ['Point of Contact_Contact Agent',
       'Point of Contact_Contact Owner', 'BHK', 'City_Mumbai', 'Size',
       'Area Type_Carpet Area', 'Area Type_Super Area', 'City_Kolkata',
       'Furnishing Status_Unfurnished', 'Floor Category_Ground',
       'Floor Category_Low', 'Furnishing Status_Furnished',
       'Furnishing Status_Semi-Furnished', 'Season_Summer', 'Season_Monsoon',
       'Floor Category_Medium', 'City_Bangalore', 'City_Chennai',
       'City_Hyderabad', 'Point of Contact_Contact Builder',
       'Floor Category_Basement', 'City_Delhi', 'Floor Category_High',
       'Area Type_Built Area']

In [63]:
X = df[features]  # Ознаки
y = df['Rent_log']  # Цільова змінна

print(f"\nРозмір X (ознак): {X.shape}")
print(f"Розмір y (цілі): {y.shape}")


Розмір X (ознак): (4226, 24)
Розмір y (цілі): (4226,)


In [64]:
from sklearn.preprocessing import StandardScaler

In [65]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

## Завдання 5: Розділення даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на навчальну (80%) та тестову (20%) вибірки.
2. Створіть модель лінійної регресії.
3. Навчіть модель на навчальних даних.
4. Виведіть усі коефіцієнти моделі (ваги) та напишіть, які 2 ознаки найбільше впливають на прогноз.
5. Зробіть прогнози на тренувальній та тестовій вибірках.

In [66]:
from sklearn.model_selection import train_test_split

In [67]:
# Розділяємо дані: 80% на навчання, 20% на тест
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y,
    test_size=0.2,
    random_state=42
)

In [68]:
from sklearn.linear_model import LinearRegression

# Створюємо модель
model = LinearRegression()

# Навчаємо модель на навчальних даних
model.fit(X_train, y_train)

LinearRegression()

In [70]:
feature_weights = list(zip(model.feature_names_in_, model.coef_))

# Sort the list in descending order based on the weight
sorted_feature_weights = sorted(feature_weights, key=lambda x: x[1], reverse=True)

for feature, weight in sorted_feature_weights:
    print(f"{feature}: {weight:.2f}")

print(f"\nЗміщення (intercept): {model.intercept_:.2f}")

City_Mumbai: 0.27
BHK: 0.21
Size: 0.21
Point of Contact_Contact Agent: 0.07
Furnishing Status_Furnished: 0.06
Floor Category_Low: 0.02
City_Delhi: 0.02
Area Type_Carpet Area: 0.01
Furnishing Status_Semi-Furnished: 0.01
Season_Monsoon: 0.00
Floor Category_Basement: 0.00
Season_Summer: -0.00
Area Type_Built Area: -0.00
Point of Contact_Contact Builder: -0.01
Area Type_Super Area: -0.01
City_Bangalore: -0.03
Floor Category_High: -0.03
Floor Category_Medium: -0.04
City_Chennai: -0.04
Furnishing Status_Unfurnished: -0.05
Floor Category_Ground: -0.06
Point of Contact_Contact Owner: -0.07
City_Hyderabad: -0.07
City_Kolkata: -0.12

Зміщення (intercept): 9.64


In [74]:
# Прогнози на навчальній вибірці
y_train_pred = model.predict(X_train)

# Прогнози на тестовій вибірці (нові дані!)
y_test_pred = model.predict(X_test)

# Перетворення з log назад на оригінальний масштаб
y_test_pred_actual = np.exp(y_test_pred)  # Apply exponential function to get back to original scale
y_test_actual = np.exp(y_test)  # Apply exponential function to get back to original scale

# Порівняння перших 10 прогнозів з реальністю
comparison = pd.DataFrame({
    'Реальна оренда': y_test_actual[:10].round(0),  # Round to nearest integer for better readability
    'Прогнозована оренда': y_test_pred_actual[:10].round(0),  # Round the predictions as well
    'Помилка': (y_test_actual[:10] - y_test_pred_actual[:10]).round(0)  # Calculate error
})

print("Приклади прогнозів на тестовій вибірці:")
print(comparison)

Приклади прогнозів на тестовій вибірці:
      Реальна оренда  Прогнозована оренда  Помилка
1998         22000.0              28335.0  -6335.0
3190          5000.0               6774.0  -1774.0
2670         37000.0              43630.0  -6630.0
4647          8000.0               6791.0   1209.0
4488         15000.0              13256.0   1744.0
2672         20000.0              23097.0  -3097.0
2802          8500.0              13092.0  -4592.0
352           7000.0               6242.0    758.0
221           3000.0               5179.0  -2179.0
491           8000.0               8796.0   -796.0


## Завдання 6: Оцінка якості моделі (2 бали)

**Що потрібно зробити:**
1. Обчисліть MAE, RMSE та R² для навчальної та тестової вибірок
2. Порівняйте метрики та зробіть висновок про якість моделі
3. Проаналізуйте і дайте висновок, чи є ознаки перенавчання або недонавчання (**Нагадування**: перенавчання - коли модель дуже добре працює на тренувальних даних, але погано на тестових; недонавчання - коли модель погано працює навіть на тренувальних даних)
4. Побудуйте графік розсіювання "реальні vs прогнозовані значення" та зробіть висновок про якість моделі


In [75]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Inverse the log transformation (exp) to get back to original scale
y_test_pred_actual = np.exp(y_test_pred)
y_test_actual = np.exp(y_test)

y_train_pred_actual = np.exp(y_train_pred)
y_train_actual = np.exp(y_train)

# Calculate MAE, MSE, RMSE, R² for the test set
mae = mean_absolute_error(y_test_actual, y_test_pred_actual)
mse = mean_squared_error(y_test_actual, y_test_pred_actual)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_actual, y_test_pred_actual)

# Print the metrics for the test set
print("="*50)
print("МЕТРИКИ ЯКОСТІ МОДЕЛІ (на тестовій вибірці):")
print("="*50)
print(f"\nMAE: {mae:.2f} рупій")
print(f"RMSE: {rmse:.2f} рупій")
print(f"R²: {r2:.3f}")

# Calculate MAE, MSE, RMSE, R² for the training set
mae_train = mean_absolute_error(y_train_actual, y_train_pred_actual)
mse_train = mean_squared_error(y_train_actual, y_train_pred_actual)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train_actual, y_train_pred_actual)

# Print the metrics for the training set
print("="*50)
print("МЕТРИКИ ЯКОСТІ МОДЕЛІ на тренувальній вибірці:")
print("="*50)
print(f"\nMAE: {mae_train:.2f} рупій")
print(f"RMSE: {rmse_train:.2f} рупій")
print(f"R²: {r2_train:.3f}")

МЕТРИКИ ЯКОСТІ МОДЕЛІ (на тестовій вибірці):

MAE: 5292.46 рупій
RMSE: 8539.71 рупій
R²: 0.620
МЕТРИКИ ЯКОСТІ МОДЕЛІ на тренувальній вибірці:

MAE: 5198.80 рупій
RMSE: 8200.41 рупій
R²: 0.648


In [79]:
np.mean(np.exp(y_train)).round(2)

np.float64(19349.55)

In [81]:
ratio = 5292.46 / 19349.55
print(f"Ratio of MAE to mean rent: {ratio:.3f}")

Ratio of MAE to mean rent: 0.274


Значення MAE показують середню абсолютну помилку в передбаченнях орендної плати. Помилка на тестовій вибірці трохи більша, ніж на навчальній, що вказує на те, що модель може працювати дещо гірше на нових, невідомих даних. RMSE також вказує на величину помилки в прогнозах, однак ця метрика більш чутлива до великих відхилень. В нашому випадку, різниця між R² на навчальній вибірці (0.648) і тестовій вибірці (0.620) не є значною, що означає, що модель не є сильно перенавченою. Модель працює добре на навчальних даних, але не має надмірної спеціалізації на них. Помилка 27.4% (на основі MAE відносно середньої оренди) свідчить про те, що модель робить помірні помилки. Хоча це не катастрофічно, помилка все одно велика.

In [83]:
fig = px.scatter(
    x=y_test_actual,
    y=y_test_pred_actual,
    title='Реальні vs Прогнозовані ціни на оренду (тестова вибірка)',
    labels={'x': 'Реальна ціна', 'y': 'Прогнозована ціна'},
    opacity=0.6
)

# Додаємо ідеальну лінію (де прогноз = реальність)
max_val = max(y_test_actual.max(), y_test_pred_actual.max())
fig.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        name='Ідеальний прогноз',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(height=500)
fig.show()

## Завдання 7: Аналіз помилок (4 бали)

**Що потрібно зробити:**
1. Обчисліть помилки (residuals = реальні - прогнозовані значення)
2. Побудуйте гістограму розподілу помилок
3. Створіть scatter plot помилок відносно величини прогнозованих значень. Чи росте помилка з ростом прогнозованого значення?
4. Знайдіть 5 прогнозів з найбільшими помилками
5. Проаналізуйте, на яких типах житла модель помиляється найбільше. Типи можна розрізняти за кількістю кімнат чи містом, наприклад.
6. Подумайте і напишіть, які наступні кроки ви б зробили, аби поліпшити якість моделі. Опціонально можна їх зробити і ми перевіримо :)

In [86]:
# Розраховуємо помилки (залишки)
residuals = y_test_actual - y_test_pred_actual

# Гістограма помилок
fig = px.histogram(
    x=residuals,
    nbins=50,
    title='Розподіл помилок прогнозування',
    labels={'x': 'Помилка (реальні - прогнозовані)'},
    color_discrete_sequence=['#e74c3c']
)
fig.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Ідеальний прогноз")
fig.update_layout(
    yaxis_title='Кількість оголошень')
fig.update_layout(height=400)
fig.show()

Більшість помилок прогнозування сконцентрована навколо нуля, що вказує на те, що більшість прогнозів досить близькі до реальних значень. Це свідчить про те, що модель загалом працює добре для більшості випадків.
Нахил: Є помітний правий нахил гістограми, що означає, що більше помилок знаходяться на позитивній стороні (де прогноз нижчий за реальне значення), в той час як відносно менше випадків, коли прогноз перевищує реальне значення.
Викиди: Є кілька екстремальних значень (як позитивних, так і негативних), хоча вони з'являються рідше, у хвостах розподілу. Це може свідчити про наявність значних викидів, де прогнози моделі були сильно неточними.

In [85]:
# Scatter plot: помилки vs прогнозовані значення
fig = px.scatter(
    x=y_test_pred,
    y=residuals,
    title='Залежність помилок від прогнозованих значень',
    labels={'x': 'Прогнозована ціна', 'y': 'Помилка'},
    opacity=0.5
)

# Додаємо горизонтальну лінію на 0
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Без помилки")

fig.update_layout(height=400)
fig.show()

Розсіювання точок досить рівномірне, без явного патерну чи тренду. Більшість точок розташовані близько до нуля по осі y, що вказує на те, що помилки дуже близькі до нуля. У другій частині scatter plot, де прогнозовані значення наближаються до більш високих значень, помилки стають більш розсіяними. Це свідчить про те, що з ростом прогнозованих значень (по осі X) розмір помилок варіюється значно більше, ніж на початку (ліву частину графіку).

In [89]:
top_5_errors = comparison['Помилка'].abs().nlargest(5)

# Вивести 5 прогнозів з найбільшими помилками
top_5_comparison = comparison.loc[top_5_errors.index]

print("5 прогнозів з найбільшими помилками:")
print(top_5_comparison)

5 прогнозів з найбільшими помилками:
      Реальна оренда  Прогнозована оренда  Помилка
2670         37000.0              43630.0  -6630.0
1998         22000.0              28335.0  -6335.0
2802          8500.0              13092.0  -4592.0
2672         20000.0              23097.0  -3097.0
221           3000.0               5179.0  -2179.0


In [100]:
df = X_test.copy()
df['error'] = residuals
df['abs_error'] = np.abs(residuals)  # absolute error to avoid +/- cancellation

categorical_columns = [
    'Point of Contact_Contact Agent', 'Point of Contact_Contact Owner', 'BHK', 'City_Mumbai',
    'Area Type_Carpet Area', 'Area Type_Super Area', 'City_Kolkata', 'Furnishing Status_Unfurnished',
    'Floor Category_Ground', 'Floor Category_Low', 'Furnishing Status_Furnished',
    'Furnishing Status_Semi-Furnished', 'Season_Summer', 'Season_Monsoon', 'Floor Category_Medium',
    'City_Bangalore', 'City_Chennai', 'City_Hyderabad', 'Point of Contact_Contact Builder',
    'Floor Category_Basement', 'City_Delhi', 'Floor Category_High', 'Area Type_Built Area'
]

# Build a flat list of (category_label, mean_abs_error, mean_signed_error, count)
rows = []

for column in categorical_columns:
    stats = df.groupby(column)['abs_error'].agg(['mean', 'count'])
    signed = df.groupby(column)['error'].mean()

    for group_val, row in stats.iterrows():
        # Only include the "positive" group for binary 0/1 columns (where value == 1 means "yes")
        if set(df[column].unique()).issubset({0, 1}):
            if group_val == 0:
                continue  # skip the "not this category" group

        rows.append({
            'Category': f"{column} = {group_val}",
            'Mean Abs Error': row['mean'],
            'Mean Signed Error': signed[group_val],
            'Count': int(row['count'])
        })

result = pd.DataFrame(rows).sort_values('Mean Abs Error', ascending=False)

print("=" * 30)
print(f"{'АНАЛІЗ ПОМИЛОК ЗА ТИПАМИ ЖИТЛА':^30}")
print(f"{'(відсортовано від найбільшої до найменшої середньої абсолютної помилки)':^30}")
print("=" * 30)
print(f"{'Категорія':<40} {'Сер. |помилка|':>14} {'Сер. помилка':>12} {'N':>5}")
print("-" * 30)

for _, row in result.iterrows():
    direction = "↑ завищує" if row['Mean Signed Error'] > 0 else "↓ занижує"
    print(f"{row['Category']:<40} {row['Mean Abs Error']:>14,.0f} {row['Mean Signed Error']:>+12,.0f}  ({direction})  n={row['Count']}")

print("=" * 30)
print(f"\nТоп-5 категорій з НАЙБІЛЬШОЮ помилкою:")
print(result.head(5)[['Category', 'Mean Abs Error', 'Mean Signed Error', 'Count']].to_string(index=False))

АНАЛІЗ ПОМИЛОК ЗА ТИПАМИ ЖИТЛА
(відсортовано від найбільшої до найменшої середньої абсолютної помилки)
Категорія                                Сер. |помилка| Сер. помилка     N
------------------------------
BHK = 4.073931478300406                          30,735      -30,735  (↓ занижує)  n=1
BHK = 5.414253348856849                          22,161      -22,161  (↓ занижує)  n=2
BHK = 2.7336096077439627                         15,686      -11,231  (↓ занижує)  n=17
City_Mumbai = 2.4464513007501596                  9,759         +343  (↑ завищує)  n=118
Furnishing Status_Furnished = 2.6696297640165527          9,518       -1,073  (↓ занижує)  n=95
Point of Contact_Contact Owner = -1.7304129340984695          9,081         +570  (↑ завищує)  n=213
Point of Contact_Contact Agent = 1.7315045054554399          9,081         +570  (↑ завищує)  n=213
BHK = 1.3932877371875192                          8,008       +1,633  (↑ завищує)  n=152
City_Delhi = 2.579883156549783                    7,54

Для покращення:

In [ ]:
# Ціна за квадратний метр — дуже сильна ознака
df['price_per_sqm'] = df['Rent'] / df['Size']

# Співвідношення площі до кімнат — виявляє незвично малі/великі кімнати
df['sqm_per_room'] = df['Size'] / df['BHK']

# Взаємодія: Місто + Меблювання (меблі в Мумбаї ≠ меблі в Калькутті)
df['City_Furnishing'] = df['City'] + '_' + df['Furnishing Status']

Бустингові моделі зазвичай набагато краще справляються з прогнозуванням оренди, бо залежності нелінійні

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)

RMSE=8539 значно більше за MAE=5292 — такий розрив зазвичай вказує на викиди, які псують модель.